In [1]:
import pandas as pd
import numpy as np
import json

def process_job_data(file_path):
    """
    Comprehensive data loading and cleaning pipeline for 1M+ records.
    Ensures all columns are retained and optimized for downstream insights.
    """
    # 1. Complete optimal data types to minimize memory across all features
    optimized_dtypes = {
        'employmentTypes': 'category',
        'metadata_isPostedOnBehalf': 'boolean',
        'metadata_jobPostId': 'string',
        'metadata_repostCount': 'Int16',
        'metadata_totalNumberJobApplication': 'Int32', 
        'metadata_totalNumberOfView': 'Int32',
        'minimumYearsExperience': 'Int8',
        'numberOfVacancies': 'Int16',
        'positionLevels': 'category',
        'postedCompany_name': 'string',
        'salary_maximum': 'float32',
        'salary_minimum': 'float32',
        'salary_type': 'category',
        'status_jobStatus': 'category',
        'title': 'string',
        'average_salary': 'float32'
    }
    
    # 2. Define date columns for automatic parsing (vital for time-series insights)
    date_cols = [
        'metadata_expiryDate', 
        'metadata_newPostingDate', 
        'metadata_originalPostingDate'
    ]

    print("Loading data...")
    # 3. Load the CSV file
    df = pd.read_csv(
        file_path,
        dtype=optimized_dtypes,
        parse_dates=date_cols,
        dayfirst=True,  # Ensures dates like 22/4/2023 are parsed correctly
        low_memory=False,
        #nrows = 50000
    )
    
    # 4. Drop columns that are completely empty
    if 'occupationId' in df.columns:
        df = df.drop(columns=['occupationId'])
        print("Dropped empty 'occupationId' column.")

    # 5. Handle extreme outliers identified in descriptive stats
    print("Analyzing extreme outliers...")
    initial_len = len(df)
    
    # Step A: Identify the bad rows (Experience > 50 OR Salary > 1,000,000)
    exp_outliers_mask = df['minimumYearsExperience'] > 50
    salary_outliers_mask = df['salary_maximum'] > 1000000
    
    # Combine the masks using | (OR) to catch rows that have either issue (or both)
    combined_bad_rows_mask = exp_outliers_mask | salary_outliers_mask
    
    # Step B: Count and print the impact before deleting
    print(f"Rows with > 50 years experience: {exp_outliers_mask.sum()}")
    print(f"Rows with > 1000000 max salary: {salary_outliers_mask.sum()}")
    print(f"Total unique rows to be deleted: {combined_bad_rows_mask.sum()}")
    
    # Step C: Actually remove the bad rows 
    # The tilde (~) operator flips the mask to keep the GOOD rows
    df = df[~combined_bad_rows_mask]
    
    print(f"Verification: Removed {initial_len - len(df)} rows.")

    # 6. Parse the JSON string in 'categories'
    print("Parsing JSON categories...")
    def extract_categories(json_str):
        if pd.isna(json_str):
            return []
        try:
            parsed = json.loads(json_str)
            return [item.get('category') for item in parsed if 'category' in item]
        except (json.JSONDecodeError, TypeError):
            return []

    # Apply extraction and drop the original raw JSON string column
    df['parsed_categories'] = df['categories'].apply(extract_categories)
    df = df.drop(columns=['categories'])

    # 7. Drop invalid "ghost" records
    # If a job has no title or no Job ID, it's useless for analysis.
    df = df.dropna(subset=['title', 'metadata_jobPostId'])
    print(f"Dropped rows missing critical identifiers. New row count: {len(df)}")
    
    print("Pipeline execution complete. DataFrame is ready for analysis.")
    return df

# Execute the function and store it in memory for all subsequent cells
# Replace 'jobs_dataset.csv' with your actual file
df_clean = process_job_data('SGJobData.csv')

# Verify the fully loaded and cleaned dataframe
df_clean.info()


Loading data...


C:\Users\nived\AppData\Local\Temp\ipykernel_35624\3287316107.py:39: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df = pd.read_csv(


Dropped empty 'occupationId' column.
Analyzing extreme outliers...
Rows with > 50 years experience: 17
Rows with > 1000000 max salary: 12
Total unique rows to be deleted: 22
Verification: Removed 22 rows.
Parsing JSON categories...
Dropped rows missing critical identifiers. New row count: 1044575
Pipeline execution complete. DataFrame is ready for analysis.
<class 'pandas.DataFrame'>
Index: 1044575 entries, 0 to 1048574
Data columns (total 21 columns):
 #   Column                              Non-Null Count    Dtype         
---  ------                              --------------    -----         
 0   employmentTypes                     1044575 non-null  category      
 1   metadata_expiryDate                 1044575 non-null  str           
 2   metadata_isPostedOnBehalf           1044575 non-null  boolean       
 3   metadata_jobPostId                  1044575 non-null  string        
 4   metadata_newPostingDate             1044575 non-null  str           
 5   metadata_originalPos

Cell2: Mismatch cell analysis

In [2]:
# 1. Explode the pre-parsed categories so each job can be counted under multiple sectors if necessary
df_exploded = df_clean.explode('parsed_categories')

# 2. Aggregate Market Signals by Sector
print("Calculating supply and demand metrics...")
sector_metrics = df_exploded.groupby('parsed_categories').agg(
    total_postings=('metadata_jobPostId', 'count'),
    total_vacancies=('numberOfVacancies', 'sum'),
    total_applications=('metadata_totalNumberJobApplication', 'sum'),
    avg_repost_count=('metadata_repostCount', 'mean'),
    avg_salary=('average_salary', 'mean'),
    avg_years_exp=('minimumYearsExperience', 'mean')
).reset_index()

# 3. Statistical Significance Filter ---
# Filter out niche sectors with too few postings to provide reliable metrics
# (You can adjust the '30' threshold based on your needs)
sector_metrics = sector_metrics[sector_metrics['total_postings'] >= 30]

# 4. Calculate Critical Efficiency Ratios
# Applications per Vacancy (Supply Density)
sector_metrics['apps_per_vacancy'] = (
    sector_metrics['total_applications'] / sector_metrics['total_vacancies']
).replace([np.inf, -np.inf], np.nan)

# 5. Identify Shortages (High Reposting, Low Supply)
# Sorting by highest repost count and lowest applications per vacancy
shortages = sector_metrics.sort_values(
    by=['avg_repost_count', 'apps_per_vacancy'], 
    ascending=[False, True]
).dropna(subset=['apps_per_vacancy'])

# 6. Display Results
columns_to_display = [
    'parsed_categories', 'total_vacancies', 'avg_repost_count', 
    'apps_per_vacancy', 'avg_salary'
]
print("\n--- Top 10 Sectors Experiencing Critical Skills Gaps ---")
print(shortages[columns_to_display].head(10).to_markdown(index=False))


Calculating supply and demand metrics...

--- Top 10 Sectors Experiencing Critical Skills Gaps ---
| parsed_categories          |   total_vacancies |   avg_repost_count |   apps_per_vacancy |   avg_salary |
|:---------------------------|------------------:|-------------------:|-------------------:|-------------:|
| Purchasing / Merchandising |             27363 |          0.163178  |           1.49834  |      4511.32 |
| Manufacturing              |            121974 |          0.118311  |           0.79512  |      4239.15 |
| Telecommunications         |             67154 |          0.0966972 |           0.235474 |      5032.62 |
| Engineering                |            259199 |          0.085862  |           0.884131 |      4996.15 |
| Hospitality                |             93529 |          0.08282   |           0.471137 |      3379.3  |
| Precision Engineering      |             10700 |          0.0827492 |           0.932804 |      4416.08 |
| Others                     |       

In [17]:
print(df_clean.columns.tolist())

['employmentTypes', 'metadata_expiryDate', 'metadata_isPostedOnBehalf', 'metadata_jobPostId', 'metadata_newPostingDate', 'metadata_originalPostingDate', 'metadata_repostCount', 'metadata_totalNumberJobApplication', 'metadata_totalNumberOfView', 'minimumYearsExperience', 'numberOfVacancies', 'positionLevels', 'postedCompany_name', 'salary_maximum', 'salary_minimum', 'salary_type', 'status_id', 'status_jobStatus', 'title', 'average_salary', 'parsed_categories']


In [3]:
# Average salary for each job category
salary_by_sector = (
    df_clean.explode('parsed_categories')
    .groupby('parsed_categories')['average_salary']
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)

print("\n===== Highest Paying Job Categories =====")
print(salary_by_sector.head(10).to_markdown(index=False))


===== Highest Paying Job Categories =====
| parsed_categories      |   average_salary |
|:-----------------------|-----------------:|
| Risk Management        |          7464.6  |
| Banking and Finance    |          7027.72 |
| Information Technology |          7010.58 |
| Consulting             |          6195.22 |
| Insurance              |          5917.81 |
| Legal                  |          5824.97 |
| General Management     |          5482.79 |
| Professional Services  |          5356.35 |
| Telecommunications     |          5032.62 |
| Engineering            |          4996.15 |


In [4]:
# Most viewed job categories
views_by_sector = (
    df_clean.explode('parsed_categories')
    .groupby('parsed_categories')['metadata_totalNumberOfView']
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)

print("\n===== Most Viewed Job Categories =====")
print(views_by_sector.head(10).to_markdown(index=False))


===== Most Viewed Job Categories =====
| parsed_categories                 |   metadata_totalNumberOfView |
|:----------------------------------|-----------------------------:|
| Social Services                   |                      42.7101 |
| Risk Management                   |                      38.5553 |
| Banking and Finance               |                      37.6779 |
| Design                            |                      34.6419 |
| Accounting / Auditing / Taxation  |                      34.5591 |
| Travel / Tourism                  |                      34.2202 |
| Wholesale Trade                   |                      33.4521 |
| Security and Investigation        |                      32.5109 |
| Public / Civil Service            |                      32.1739 |
| Real Estate / Property Management |                      32.0269 |


In [5]:
experience_by_sector = (
    df_clean.explode('parsed_categories')
    .groupby('parsed_categories')['minimumYearsExperience']
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)

print("\n===== Average Years of Experience Required =====")
print(experience_by_sector.head(10).to_markdown(index=False))


===== Average Years of Experience Required =====
| parsed_categories                 |   minimumYearsExperience |
|:----------------------------------|-------------------------:|
| Risk Management                   |                  4.2934  |
| Information Technology            |                  3.81243 |
| Building and Construction         |                  3.68737 |
| General Management                |                  3.52767 |
| Banking and Finance               |                  3.44397 |
| Real Estate / Property Management |                  3.27314 |
| Architecture / Interior Design    |                  3.2321  |
| Consulting                        |                  3.21912 |
| Engineering                       |                  3.18302 |
| Professional Services             |                  3.00178 |


In [6]:
#Field categories with the biggest talent shortages
high_demand = (
    df_clean.explode('parsed_categories')
    .groupby('parsed_categories')
    .agg(
        total_vacancies=('numberOfVacancies', 'sum'),
        total_applications=('metadata_totalNumberJobApplication', 'sum'),
        avg_reposts=('metadata_repostCount', 'mean')
    )
)

high_demand['applications_per_vacancy'] = (
    high_demand['total_applications'] /
    high_demand['total_vacancies']
)

high_demand = high_demand.sort_values(
    by=['total_vacancies', 'applications_per_vacancy'],
    ascending=[False, True]
)

print(high_demand.head(10))

                             total_vacancies  total_applications  avg_reposts  \
parsed_categories                                                               
Customer Service                      415170              195714     0.063089   
Information Technology                315784              354150     0.054543   
F&B                                   308650               65871     0.064722   
Others                                298785              180923      0.07758   
Admin / Secretarial                   292044              311267     0.044371   
Sales / Retail                        283564              145609     0.049428   
Engineering                           259199              229166     0.085862   
Healthcare / Pharmaceutical           229267               89179     0.059689   
Education and Training                222040               79669     0.063529   
Human Resources                       193274              142592     0.036249   

                           

In [22]:
#Most in demand sectors 
demand_by_sector = (
    df_clean.explode('parsed_categories')
    .groupby('parsed_categories')
    .agg(
        total_jobs=('metadata_jobPostId','count'),
        total_vacancies=('numberOfVacancies','sum')
    )
    .sort_values(
        by='total_vacancies',
        ascending=False
    )
)

print("===== Most In-Demand Sectors =====")
print(demand_by_sector.head(10))

===== Most In-Demand Sectors =====
                             total_jobs  total_vacancies
parsed_categories                                       
Customer Service                 111778           415170
Information Technology           140862           315784
F&B                               73731           308650
Others                           106097           298785
Admin / Secretarial              117848           292044
Sales / Retail                   105061           283564
Engineering                      136370           259199
Healthcare / Pharmaceutical       52740           229267
Education and Training            44877           222040
Human Resources                   53463           193274


In [23]:
print("===== Singapore Job Market Summary =====")

print(f"Total Job Postings: {df_clean['metadata_jobPostId'].nunique():,}")
print(f"Total Companies: {df_clean['postedCompany_name'].nunique():,}")
print(f"Total Vacancies: {df_clean['numberOfVacancies'].sum():,}")
print(f"Average Salary: ${df_clean['average_salary'].mean():,.2f}")
print(f"Average Applications per Job: {df_clean['metadata_totalNumberJobApplication'].mean():.2f}")
print(f"Average Views per Job: {df_clean['metadata_totalNumberOfView'].mean():.2f}")

===== Singapore Job Market Summary =====
Total Job Postings: 1,044,575
Total Companies: 53,151
Total Vacancies: 2,804,635
Average Salary: $4,712.59
Average Applications per Job: 2.14
Average Views per Job: 26.82
